In [1]:
# ===================================================================
# SISTEMA DE ANÁLISIS AMBIENTAL PARA AULA 0.13
# ===================================================================
# AUTOR: Carmen Gómez Alarcón
# FECHA: 2026
# FASE: Paso 5 v2 — Regresión con modelos ampliados
# ===================================================================

import pandas as pd
import numpy as np
import os
import pickle
from sklearn.linear_model import LinearRegression, Ridge, Lasso, ElasticNet
from sklearn.ensemble import (RandomForestRegressor, GradientBoostingRegressor,
                               ExtraTreesRegressor, HistGradientBoostingRegressor)
from sklearn.tree import DecisionTreeRegressor
from sklearn.svm import SVR
from sklearn.neighbors import KNeighborsRegressor  # ← AÑADIDO KNN
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
import warnings
warnings.filterwarnings('ignore')

# XGBoost y LightGBM (opcionales)
try:
    from xgboost import XGBRegressor
    HAS_XGB = True
except ImportError:
    HAS_XGB = False

try:
    from lightgbm import LGBMRegressor
    HAS_LGB = True
except ImportError:
    HAS_LGB = False

# ==========================================
# CONFIGURACIÓN
# ==========================================
INPUT_DIR     = 'ml_normalized_grouped'
OUTPUT_DIR    = 'ml_results_grouped'
os.makedirs(OUTPUT_DIR, exist_ok=True)

# ==========================================
# 1. CARGAR DATOS
# ==========================================
print("\n1. LOADING DATA...")

X_train_norm = pd.read_excel(os.path.join(INPUT_DIR, 'X_train_normalised.xlsx')).values
X_test_norm  = pd.read_excel(os.path.join(INPUT_DIR, 'X_test_normalised.xlsx')).values
X_train_raw  = pd.read_excel(os.path.join(INPUT_DIR, 'X_train.xlsx')).values
X_test_raw   = pd.read_excel(os.path.join(INPUT_DIR, 'X_test.xlsx')).values
y_train      = pd.read_excel(os.path.join(INPUT_DIR, 'y_train.xlsx')).values.ravel()
y_test       = pd.read_excel(os.path.join(INPUT_DIR, 'y_test.xlsx')).values.ravel()

with open(os.path.join(INPUT_DIR, 'selected_features.pkl'), 'rb') as f:
    selected_features = pickle.load(f)

STD_Y = y_train.std()
print(f"   X_train: {X_train_raw.shape} | X_test: {X_test_raw.shape}")
print(f"   y_train: {len(y_train)} | y_test: {len(y_test)}")
print(f"   STD_Y: {STD_Y:.2f}")

# ==========================================
# 2. DIVISIÓN INTERNA 80/20
# ==========================================
print("\n2. SPLITTING TRAINING DATA (80/20)...")

split_idx = int(len(y_train) * 0.8)

X_train_raw_80  = X_train_raw[:split_idx]
X_train_raw_20  = X_train_raw[split_idx:]
X_train_norm_80 = X_train_norm[:split_idx]
X_train_norm_20 = X_train_norm[split_idx:]
y_train_80      = y_train[:split_idx]
y_train_20      = y_train[split_idx:]

print(f"   Train: {len(y_train_80)} | Val: {len(y_train_20)}")

# ==========================================
# 3. DEFINIR MODELOS
# ==========================================
print("\n3. DEFINING MODELS...")

needs_scaling = {'Linear Regression', 'Ridge', 'Lasso', 'ElasticNet', 'SVR', 'KNN'}  # ← KNN AÑADIDO

models = {
    'Linear Regression': LinearRegression(),
    'Ridge':             Ridge(alpha=1.0, random_state=42),
    'Lasso':             Lasso(alpha=0.1, random_state=42),
    'ElasticNet':        ElasticNet(alpha=0.1, l1_ratio=0.5, random_state=42),
    'KNN':               KNeighborsRegressor(n_neighbors=5, weights='distance', n_jobs=-1),  # ← AÑADIDO
    'Decision Tree':     DecisionTreeRegressor(max_depth=10, min_samples_split=5, random_state=42),
    'Random Forest':     RandomForestRegressor(n_estimators=200, max_depth=15, random_state=42, n_jobs=-1),
    'Extra Trees':       ExtraTreesRegressor(n_estimators=200, max_depth=15, random_state=42, n_jobs=-1),
    'Gradient Boosting': GradientBoostingRegressor(n_estimators=200, max_depth=5, random_state=42),
    'HistGradBoost':     HistGradientBoostingRegressor(max_iter=200, max_depth=6, random_state=42),
    'SVR':               SVR(kernel='rbf', C=10, gamma='scale'),
}

if HAS_XGB:
    models['XGBoost'] = XGBRegressor(
        n_estimators=300, max_depth=6, learning_rate=0.05,
        subsample=0.8, colsample_bytree=0.8, random_state=42,
        n_jobs=-1, verbosity=0
    )

if HAS_LGB:
    models['LightGBM'] = LGBMRegressor(
        n_estimators=300, max_depth=6, learning_rate=0.05,
        subsample=0.8, colsample_bytree=0.8, random_state=42,
        n_jobs=-1, verbose=-1
    )

print(f"   Modelos: {len(models)}")

# ==========================================
# 4. FUNCIONES DE MÉTRICAS ADICIONALES
# ==========================================
def accuracy_headcount(y_true, y_pred, tolerance=0):
    return np.mean(np.abs(y_true - y_pred) <= tolerance)

def mape(y_true, y_pred):
    mask = y_true != 0
    if mask.sum() == 0:
        return 0
    return np.mean(np.abs((y_true[mask] - y_pred[mask]) / y_true[mask])) * 100

def smape(y_true, y_pred):
    denom = (np.abs(y_true) + np.abs(y_pred)) / 2
    mask = denom != 0
    if mask.sum() == 0:
        return 0
    return np.mean(np.abs(y_true[mask] - y_pred[mask]) / denom[mask]) * 100

def max_error(y_true, y_pred):
    return np.max(np.abs(y_true - y_pred))

def explained_variance(y_true, y_pred):
    from sklearn.metrics import explained_variance_score
    return explained_variance_score(y_true, y_pred)

def diagnosticar(train_rmse, val_rmse, std_y):
    gap = val_rmse - train_rmse
    gap_ratio = gap / train_rmse if train_rmse > 0 else 0
    if train_rmse > 1.5 * std_y and val_rmse > 1.5 * std_y:
        return 'UNDERFITTING SEVERO', gap_ratio
    if train_rmse > 1.0 * std_y and val_rmse > 1.0 * std_y:
        return 'UNDERFITTING', gap_ratio
    if train_rmse < 0.4 * std_y and gap_ratio > 0.5:
        return 'OVERFITTING SEVERO', gap_ratio
    if gap_ratio > 0.35 or gap > 0.5 * std_y:
        return 'OVERFITTING', gap_ratio
    if gap_ratio > 0.20 or gap > 0.25 * std_y:
        return 'OVERFITTING LEVE', gap_ratio
    return 'BUEN AJUSTE', gap_ratio

# ==========================================
# 5. ENTRENAR Y EVALUAR
# ==========================================
print("\n4. TRAINING...")
print("-"*70)

results        = {}
trained_models = {}
predictions    = {}

for name, model in models.items():
    scaled = name in needs_scaling
    X_tr_80  = X_train_norm_80 if scaled else X_train_raw_80
    X_val_20 = X_train_norm_20 if scaled else X_train_raw_20
    X_tr_all = X_train_norm    if scaled else X_train_raw
    X_te     = X_test_norm     if scaled else X_test_raw

    print(f"\n   [{name}] ({'norm' if scaled else 'raw'})")

    # Paso A: entrenar 80% y validar 20%
    model.fit(X_tr_80, y_train_80)
    y_val_pred = np.clip(model.predict(X_val_20), 0, None)

    val_rmse = np.sqrt(mean_squared_error(y_train_20, y_val_pred))
    val_mae  = mean_absolute_error(y_train_20, y_val_pred)
    val_r2   = r2_score(y_train_20, y_val_pred)

    y_tr80_pred  = np.clip(model.predict(X_tr_80), 0, None)
    train_rmse_v = np.sqrt(mean_squared_error(y_train_80, y_tr80_pred))
    diagnostico, gap_ratio = diagnosticar(train_rmse_v, val_rmse, STD_Y)

    # Paso B: re-entrenar 100% y evaluar test
    model.fit(X_tr_all, y_train)
    y_test_pred = np.clip(model.predict(X_te), 0, None)

    test_rmse = np.sqrt(mean_squared_error(y_test, y_test_pred))
    test_mae  = mean_absolute_error(y_test, y_test_pred)
    test_r2   = r2_score(y_test, y_test_pred)
    
    test_acc_exacta = accuracy_headcount(y_test, y_test_pred, tolerance=0)
    test_acc_1      = accuracy_headcount(y_test, y_test_pred, tolerance=1)
    test_acc_2      = accuracy_headcount(y_test, y_test_pred, tolerance=2)
    test_acc_3      = accuracy_headcount(y_test, y_test_pred, tolerance=3)
    test_mape       = mape(y_test, y_test_pred)
    test_smape      = smape(y_test, y_test_pred)
    test_max_err    = max_error(y_test, y_test_pred)
    test_exp_var    = explained_variance(y_test, y_test_pred)

    results[name] = {
        'Val_MAE':    round(val_mae, 2),
        'Val_RMSE':   round(val_rmse, 2),
        'Val_R2':     round(val_r2, 4),
        'Val_std':    0.0,
        'Test_MAE':   round(test_mae, 2),
        'Test_RMSE':  round(test_rmse, 2),
        'Test_R2':    round(test_r2, 4),
        'MAE':        round(test_mae, 2),
        'Acc_Exacta': round(test_acc_exacta, 4),
        'Acc_±1':     round(test_acc_1, 4),
        'Acc_±2':     round(test_acc_2, 4),
        'Acc_±3':     round(test_acc_3, 4),
        'MAPE_%':     round(test_mape, 2),
        'SMAPE_%':    round(test_smape, 2),
        'Max_Error':  round(test_max_err, 1),
        'Exp_Var':    round(test_exp_var, 4),
        'gap_ratio':  round(gap_ratio, 3),
        'diagnostico': diagnostico
    }
    trained_models[name] = model
    predictions[name]    = y_test_pred

    print(f"      Val  → RMSE: {val_rmse:.2f} | MAE: {val_mae:.2f} | R²: {val_r2:.4f}")
    print(f"      Test → RMSE: {test_rmse:.2f} | MAE: {test_mae:.2f} | R²: {test_r2:.4f}")
    print(f"      Acc  → Exacta: {test_acc_exacta:.1%} | ±1: {test_acc_1:.1%} | ±2: {test_acc_2:.1%} | ±3: {test_acc_3:.1%}")
    print(f"      Error → MAPE: {test_mape:.1f}% | Max: {test_max_err:.0f} pers")
    print(f"      Diagnóstico: {diagnostico} (gap={gap_ratio:.2f})")

# ==========================================
# 6. RESUMEN
# ==========================================
print("\n" + "="*80)
print("5. MODELS SUMMARY (ordenado por Test_RMSE)")
print("="*80)

df_summary = (pd.DataFrame(results).T
                .sort_values('Test_RMSE', ascending=True))

cols_show = ['Val_R2', 'Val_RMSE', 'Test_R2', 'Test_MAE', 'Test_RMSE',
             'Acc_Exacta', 'Acc_±1', 'Acc_±2', 'Acc_±3',
             'MAPE_%', 'Max_Error', 'gap_ratio', 'diagnostico']
print(df_summary[cols_show].to_string())

best = df_summary.index[0]
print(f"\n   MEJOR MODELO: {best}")
print(f"   Test RMSE:     {results[best]['Test_RMSE']} personas")
print(f"   Test MAE:      {results[best]['Test_MAE']} personas")
print(f"   Test R²:       {results[best]['Test_R2']}")
print(f"   Acc Exacta:    {results[best]['Acc_Exacta']:.1%}")
print(f"   Acc ±2 pers:   {results[best]['Acc_±2']:.1%}")
print(f"   MAPE:          {results[best]['MAPE_%']:.1f}%")
print(f"   Max Error:     {results[best]['Max_Error']:.0f} personas")
print(f"   Diagnóstico:   {results[best]['diagnostico']}")

# ==========================================
# 7. GUARDAR
# ==========================================
print("\n6. SAVING...")

with open(os.path.join(OUTPUT_DIR, 'best_model.pkl'), 'wb') as f:
    pickle.dump(trained_models[best], f)

with open(os.path.join(OUTPUT_DIR, 'all_models.pkl'), 'wb') as f:
    pickle.dump(trained_models, f)

residuals = y_test - predictions[best]
df_preds = pd.DataFrame({
    'Actual':       y_test,
    'Predicted':    np.round(predictions[best], 1),
    'Error':        np.round(residuals, 1),
    'Abs_Error':    np.round(np.abs(residuals), 1),
    'Acc_Exacta':   (np.abs(residuals) == 0).astype(int),
    'Acc_±1':       (np.abs(residuals) <= 1).astype(int),
    'Acc_±2':       (np.abs(residuals) <= 2).astype(int),
    'Acc_±3':       (np.abs(residuals) <= 3).astype(int)
})
df_preds.to_excel(os.path.join(OUTPUT_DIR, 'predictions.xlsx'), index=False)

df_summary.to_excel(os.path.join(OUTPUT_DIR, 'models_summary.xlsx'))

df_all = pd.DataFrame({'Actual': y_test})
for name, y_pred in predictions.items():
    df_all[name] = np.round(y_pred, 1)
df_all.to_excel(os.path.join(OUTPUT_DIR, 'all_predictions.xlsx'), index=False)

with open(os.path.join(OUTPUT_DIR, 'results.pkl'), 'wb') as f:
    pickle.dump(results, f)
with open(os.path.join(OUTPUT_DIR, 'predictions.pkl'), 'wb') as f:
    pickle.dump(predictions, f)
np.save(os.path.join(OUTPUT_DIR, 'y_test.npy'), y_test)
np.save(os.path.join(OUTPUT_DIR, 'y_train.npy'), y_train)

print(f"\n   Guardado en {OUTPUT_DIR}/:")
for fname in sorted(os.listdir(OUTPUT_DIR)):
    size = os.path.getsize(os.path.join(OUTPUT_DIR, fname)) / 1024
    print(f"      {fname} ({size:.0f} KB)")

print("\n" + "="*80)
print("PASO 5 v2 COMPLETADO")
print("="*80)


1. LOADING DATA...
   X_train: (58, 10) | X_test: (25, 10)
   y_train: 58 | y_test: 25
   STD_Y: 21.08

2. SPLITTING TRAINING DATA (80/20)...
   Train: 46 | Val: 12

3. DEFINING MODELS...
   Modelos: 13

4. TRAINING...
----------------------------------------------------------------------

   [Linear Regression] (norm)
      Val  → RMSE: 7.03 | MAE: 4.42 | R²: 0.8455
      Test → RMSE: 15.64 | MAE: 10.14 | R²: 0.0931
      Acc  → Exacta: 24.0% | ±1: 28.0% | ±2: 36.0% | ±3: 40.0%
      Error → MAPE: 469.3% | Max: 37 pers
      Diagnóstico: BUEN AJUSTE (gap=-0.30)

   [Ridge] (norm)
      Val  → RMSE: 5.29 | MAE: 3.35 | R²: 0.9125
      Test → RMSE: 15.54 | MAE: 10.22 | R²: 0.1038
      Acc  → Exacta: 24.0% | ±1: 24.0% | ±2: 28.0% | ±3: 32.0%
      Error → MAPE: 473.6% | Max: 37 pers
      Diagnóstico: BUEN AJUSTE (gap=-0.48)

   [Lasso] (norm)
      Val  → RMSE: 5.34 | MAE: 3.43 | R²: 0.9109
      Test → RMSE: 15.39 | MAE: 10.14 | R²: 0.1218
      Acc  → Exacta: 24.0% | ±1: 24.0% | ±2: